In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

from matplotlib.gridspec import GridSpec
#from ipywidgets import interact, FloatSlider, FloatLogSlider
import warnings
warnings.filterwarnings('ignore')
import os

from scipy.ndimage import gaussian_filter

import visualize_local_new as vis
import matplotlib.colors as mcolors


# visualize paraview linecuts

In [ ]:
def get_slice(sample, coord, date, data_dir):
    """
    Returns a dict of all parameters/paths needed to load a single
    (sample, slice) combination.
    """

    cfg = dict(
        sample=sample,
        coord=coord
    )

    print(cfg)

    date = date

    if sample == '665':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 51 #nm
    elif sample == '270':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 66 #nm
    elif sample == '452':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 33 #nm
    elif sample == '105':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 50 #nm ################################ THIS IS WRONG NEED TO UPDATE

    else:
        raise ValueError(f"Unrecognized sample '{sample}'")

    file_path = os.path.join(data_dir, f"{sample}_{coord}_center_slice_{date}.csv")
    data = np.genfromtxt(file_path, delimiter=",", names=True)

    if coord == 'y':
        coord1 = data['Points0']
        coord2 = data['Points2']
        mz = data['magnetization2']
        mx = data['magnetization1'] ############# I changed this based on observation, may be wrong
        my = data['magnetization0']

    elif coord == 'x':
        coord1 = data['Points1']
        coord2 = data['Points2']
        mz = data['magnetization2']
        mx = data['magnetization1'] ############# I changed this based on observation, may be wrong
        my = data['magnetization0']

    eps = 1e-12
    mnorm = np.sqrt(mx**2 + my**2 + mz**2) # + eps
    mxn = mx / mnorm
    myn = my / mnorm
    mzn = mz / mnorm

    theta = np.arccos(np.clip(mzn, -1.0, 1.0))

    cfg['slice'] = data
    cfg['x'] = coord1
    cfg['z'] = coord2
    cfg['mx'] = mx
    cfg['my'] = my
    cfg['mz'] = mz
    cfg['mxn'] = mxn
    cfg['myn'] = myn
    cfg['mzn'] = mzn
    cfg['theta'] = theta
    cfg['mnorm'] = mnorm

    return cfg



### User set up

In [ ]:
### User set up


# -----------------------------------------------------------------------
# Scan To Analyze
# -----------------------------------------------------------------------

sample_ids = ['665', '270', '452', '105']
dates = ['11JULY26', '11JULY26', '11JULY26', '11JULY26']
#base_path = '/homes/emma/notebooks/RawData/{sample}/vectors/'
#base_path = r'C:\Users\emmab\Projects\Vector_Tomography\665_z_center_slice_1JULY26.csv'
base_path = ''


field_pairs = [('mz', 'mzn'), ('my', 'myn'), ('mx', 'mxn')]

# -----------------------------------------------------------------------
# SMOOTHING FLAG -- set this once, here, and it applies to everything
# downstream (both normalized AND un-normalized grids, for every sample).
# -----------------------------------------------------------------------
SMOOTH_DATA = True     # <-- set False to skip Gaussian smoothing entirely
SMOOTH_SIGMA = 2       # <-- Gaussian filter sigma (applied along x only)
how_many_slices = 9 #5 # (steps in percentage)

samples = {}
for sample, date in zip(sample_ids, dates):
    samples[sample] = {
        'y': get_slice(sample, 'y', date, base_path.format(sample=sample)),
        'x': get_slice(sample, 'x', date, base_path.format(sample=sample)),
        # 'z': get_slice(sample, 'z', date, base_path.format(sample=sample)),
    }

# Every cfg dict built here now stores BOTH the un-normalized ('{field}_grid')
# and normalized ('{field_n}_grid') grid for every field pair, and -- if
# SMOOTH_DATA is True -- both have already been Gaussian-smoothed (sigma =
# SMOOTH_SIGMA, along x only) before anything else touches them.
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        coords[coord] = vis.build_grids(
            data_dict, field_pairs, n_show=how_many_slices, xrange = None,
            smooth=False, sigma=SMOOTH_SIGMA
        )



# -----------------------------------------------------------------------
# Plot set up
# -----------------------------------------------------------------------
colors = plt.cm.viridis(np.linspace(0, 1, how_many_slices))

plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'lines.linewidth': 2.2,
    'axes.grid': False,
    'grid.alpha': 0.3,
})



# full profile

In [ ]:
#print(samples['270']['x'].keys())
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        vis.plot_magnitude(samples[sample][coord], colors)


# select one wall

In [ ]:
crop1_ranges = {
    '665': {'x': (495, 508), 'y': (485, 515)},
    '270': {'x': (590, 615), 'y': (575, 590)},
    '452': {'x': (515, 525), 'y': (415, 435)},
    '105': {'x': (503, 520), 'y': (540, 555)},
}

crop1 = {}
for sample, coords in samples.items():
    crop1[sample] = {}
    for coord, data_dict in coords.items():
        xmin, xmax = crop1_ranges[sample][coord]
        crop1[sample][coord] = {
            'dw': vis.build_grids(
                data_dict, field_pairs, n_show=how_many_slices,
                xrange = (xmin, xmax),
                smooth=SMOOTH_DATA, sigma=SMOOTH_SIGMA
            ),
            'xmin': xmin,
            'xmax': xmax,
        }


In [ ]:
# vis.gradient now always computes BOTH d{field}_dx (un-normalized) and
# d{field_n}_dx (normalized) in a single call -- no more `normalized=` flag.
# plot=True shows a 2x2 figure per field: [raw profile | raw gradient]
#                                          [norm profile | norm gradient]
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        if sample == '665':
            plot_grad = True
        else:
            plot_grad = False

        crop1[sample][coord]['dw'] = vis.gradient(crop1[sample][coord]['dw'], field_pairs, colors, plot=plot_grad)


# simulate walls

In [ ]:
### --- Define wall types and their parameters --- ###
# NOTE: these are idealized theoretical Bloch/Neel/Hybrid profiles (M/Ms),
# not derived from the measured data, so they are unaffected by the
# normalized-vs-un-normalized distinction used elsewhere in this notebook.
delta_demo = 2.5

xmin, xmax = 485, 515
x0 = np.linspace(0, (xmax - xmin), (xmax - xmin) + 1)
x0 = x0 - x0.mean()  # recenter around 0

wall_types = {
    'bloch':  {'psi': np.pi / 2, 'color': 'magenta'},
    'neel':   {'psi': 0,          'color': 'cyan'},
    'hybrid': {'psi': np.linspace(-np.pi /1.05, np.pi /1.1, 11), 'cmap': plt.cm.twilight_shifted},
}

for wall, cfg in wall_types.items():
    psi = cfg['psi']
    if np.isscalar(psi):
        Mx, My, Mz = vis.magnetization_components(x0, delta_demo, psi)
        cfg['Mx'], cfg['My'], cfg['Mz'] = Mx, My, Mz

        dx0 = x0[1] - x0[0]
        dMz_dx = np.gradient(Mz, dx0)
        dMy_dx = np.gradient(My, dx0)
        dMx_dx = np.gradient(Mx, dx0)
        cfg['dMx_dx'] = dMx_dx
        cfg['dMy_dx'] = dMy_dx
        cfg['dMz_dx'] = dMz_dx

    else:
        # hybrid: sweep over multiple psi values, store one array per psi
        Mx_list, My_list, Mz_list = [], [], []
        dMx_dx_list, dMy_dx_list, dMz_dx_list = [], [], []
        dx0 = x0[1] - x0[0]
        for psi_val in psi:
            Mx, My, Mz = vis.magnetization_components(x0, delta_demo, psi_val)
            Mx_list.append(Mx)
            My_list.append(My)
            Mz_list.append(Mz)
            dMz_dx_list.append(np.gradient(Mz, dx0))
            dMy_dx_list.append(np.gradient(My, dx0))
            dMx_dx_list.append(np.gradient(Mx, dx0))

        cfg['Mx'], cfg['My'], cfg['Mz'] = Mx_list, My_list, Mz_list
        cfg['dMx_dx'], cfg['dMy_dx'], cfg['dMz_dx'] = dMx_dx_list, dMy_dx_list, dMz_dx_list
        cfg['colors'] = cfg['cmap'](np.linspace(0, 1, len(psi)))


In [ ]:
for wall, cfg in wall_types.items():
    vis.plot_wall_components(wall, cfg, x0)
    vis.plot_wall_gradients(wall, cfg, x0)


# domain wall width

In [ ]:
# Functions
def fit_linecut_hyperbolic(x, Mz, z_shift, xmax, xmin, sample, label='', plot=True):
    pixel_size_nm = 8  # voxel size in nm
    x_data = x - x.mean()
    x_nm = x_data * pixel_size_nm

    colors_local = plt.cm.viridis(np.linspace(0, 1, len(z_shift)))

    dw_width = []
    dw_center = []

    fig = plt.figure(figsize=(21, 6))
    fig.suptitle(f'Hyperbolic Tan Fit of Data for range ({xmin}, {xmax}) \n {sample} ({label})', fontsize=20, y=1.0)
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.set_xlabel('cross section (nm)')
    ax1.set_ylabel(r'$M_z$' + (r'$/ M_{norm}$' if label == 'normalized' else ''))
    ax1.set_title('Line cuts of domains at different z')

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.set_xlabel('slice (%)')
    ax2.set_ylabel('DW width')
    ax2.set_title('fit DW width at different slices')

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.set_xlabel('slice (%)')
    ax3.set_ylabel('DW center')
    ax3.set_title('fit DW center at different slices')

    for j, eta_v in enumerate(z_shift):
        # Linecut
        ax1.plot(x_nm, Mz[j, :], color=colors_local[j], lw=1.5, marker='o', label=fr'{eta_v*100:.0f}%')

        # Fit with hyperbolic tangent
        hyperbolic_fit, hyperbolic_error = vis.fit_hyperbolic(x_nm, Mz[j, :])
        hyperbolic_plot = vis.hyperbolic(x_nm, *hyperbolic_fit)
        dw_width.append(hyperbolic_fit[2])
        dw_center.append(hyperbolic_fit[1])

        ax1.plot(x_nm, hyperbolic_plot, color=colors_local[j], linewidth=3, linestyle='--')

    ax1.legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc='upper right')
    ax2.plot(z_shift*100, np.abs(dw_width), marker='o')
    ax2.axhline(np.mean(np.abs(dw_width)), color='red', label=f'mean \N{GREEK CAPITAL LETTER DELTA} = {np.mean(dw_width):.2f}')
    ax2.legend()
    ax3.plot(z_shift*100, dw_center, marker='o')

    if not plot:
        plt.close(fig)
    else:
        plt.tight_layout()
        plt.show()

    return np.abs(dw_width), dw_center


In [ ]:
# Fit the hyperbolic-tangent domain-wall profile for BOTH the normalized
# (mzn_grid) and un-normalized (mz_grid) Mz components, storing both under
# dw_fits[sample][coord]['normalized'] / ['raw'].
dw_fits = {}

for sample, coords in samples.items():
    dw_fits[sample] = {}
    for coord, data_dict in coords.items():

        wall = crop1[sample][coord]['dw']

        dw_width_n, dw_center_n = fit_linecut_hyperbolic(
            wall['unique_x'], wall['mzn_grid'], wall['eta_targets'],
            crop1[sample][coord]['xmax'], crop1[sample][coord]['xmin'],
            sample=sample, label='normalized', plot=True
        )
        dw_width_raw, dw_center_raw = fit_linecut_hyperbolic(
            wall['unique_x'], wall['mz_grid'], wall['eta_targets'],
            crop1[sample][coord]['xmax'], crop1[sample][coord]['xmin'],
            sample=sample, label='un-normalized', plot=True
        )

        dw_fits[sample][coord] = {
            'normalized': {'dw_width': dw_width_n, 'dw_center': dw_center_n},
            'raw': {'dw_width': dw_width_raw, 'dw_center': dw_center_raw},
            'eta_targets': wall['eta_targets'],
        }

print(dw_fits['665']['x'].keys())
print(dw_fits['665']['y'].keys())


In [ ]:
## Compare x and y slice
sample_colors = ['gold', 'deeppink', 'limegreen', 'cyan']

## Compare sample per x and y (normalized branch, the physically bounded [-1,1] Mz)
for coord in ['x', 'y']:
    fig, ax = plt.subplots(figsize=(10, 6))
    for sample, color in zip(samples, sample_colors):
        fit = dw_fits[sample][coord]
        ax.plot(fit['eta_targets'] * 100, fit['normalized']['dw_width'], color=color, marker='o', label=f'{sample}')

    ax.set_xlabel('slice (%)')
    ax.set_ylabel('DW width (nm)')
    ax.set_title(f'Fit DW width across samples, {coord}-slice (Mz, normalized)')
    ax.legend(loc='upper right', title='sample')
    plt.tight_layout()
    plt.show()


fit theta per x in thickness

In [ ]:
### we know that theta(x) = arccos(mz), computed once from the normalized mz
### (theta isn't itself split into normalized/un-normalized -- it's always
### derived from the normalized mz in get_slice(), so there's only one theta_grid)
def plot_theta(cfg, colors):
    fig, ax = plt.subplots(figsize=(7, 7))
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    for j, eta_v in enumerate(cfg['eta_targets']):
        ax.plot(x_data, cfg['theta_grid'][j, :]/np.pi, color=colors[j], marker='o', lw=1.5, label=fr'{eta_v*100:.0f}%')

    ax.set_ylabel(r'θ(x) = arccos($m_z$) (1/π)')
    ax.set_xlabel('x (pix)')
    ax.axhline(1, color='crimson', label='sin(θ) = 0!!')

    ax.legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc='upper left')
    ax.set_title(f"Calculated theta per x \n {cfg['sample']} {cfg['coord']}-slice")


for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        plot_theta(crop1[sample][coord]['dw'], colors)


# calculate psi

In [ ]:
# vis.calc_psi now always computes BOTH branches in one call:
#   normalized:    psi_grid, mx_psi_grid, my_psi_grid
#   un-normalized: psi_raw_grid, mx_psi_raw_grid, my_psi_raw_grid
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        vis.calc_psi(crop1[sample][coord]['dw'], plot=True, colors=colors)

### also calculate psi for the full (uncropped) stack, for the final
### normalized-vs-un-normalized Neel/Bloch comparison further down
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        vis.calc_psi(samples[sample][coord], plot=False, colors=colors)


### create 5 separate quadrants per psi (evenly spaced between -1 and 1) and associate x values that fall into that quadrent with a color. Per x and its associated color, I want to replot mz, mx, and my as a function of x.

In [ ]:
def plot_by_psi_bin(cfg, select_psi=None, normalized=True, n_bins=11):
    suffix = '' if normalized else '_raw'
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    psi_over_pi = cfg[select_psi + f'psi{suffix}_grid'] / np.pi

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    z_k, y_k, x_k = (f'mzn_grid', f'myn_grid', f'mxn_grid') if normalized else ('mz_grid', 'my_grid', 'mx_grid')

    fig, ax = plt.subplots(1, 3, figsize=(21, 6))

    for j, eta_v in enumerate(cfg['eta_targets']):
        psi_slice = psi_over_pi[j, :]

        ax[0].plot(x_data, cfg[z_k][j, :], c=colors[j], linewidth=1, label=fr'{eta_v*100:.0f}%')
        ax[1].plot(x_data, cfg[y_k][j, :], c=colors[j], linewidth=1, label=fr'{eta_v*100:.0f}%')
        ax[2].plot(x_data, cfg[x_k][j, :], c=colors[j], linewidth=1, label=fr'{eta_v*100:.0f}%')

        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx]

        ax[0].scatter(x_data, cfg[z_k][j, :], c=point_colors)
        ax[1].scatter(x_data, cfg[y_k][j, :], c=point_colors)
        ax[2].scatter(x_data, cfg[x_k][j, :], c=point_colors)

    ax[0].set_ylabel(r'$M_{z}$')
    ax[1].set_ylabel(r'$M_{y}$')
    ax[2].set_ylabel(r'$M_{x}$')
    for a in ax:
        a.set_xlabel('x (pix)')

    tag = 'normalized' if normalized else 'un-normalized'
    fig.suptitle(f"{cfg['sample']} {cfg['coord']}-slice — colored by {select_psi} ψ ({tag})")

    ax[0].legend(fontsize=6, ncol=2, loc='upper right', title='Fraction of local thickness')
    ax[1].legend(fontsize=6, ncol=2, loc='upper right', title='Fraction of local thickness')
    ax[2].legend(fontsize=6, ncol=2, loc='upper right', title='Fraction of local thickness')

    plt.tight_layout()
    plt.show()

# not called by default (kept available, matching the original notebook's usage):
# for sample, coords in crop1.items():
#     for coord, crop_info in coords.items():
#         plot_by_psi_bin(crop_info['dw'], select_psi='', normalized=True)
#         plot_by_psi_bin(crop_info['dw'], select_psi='', normalized=False)


In [ ]:
def plot_gradient_by_psi_bin(cfg, select_psi=None, normalized=True, n_bins=11):
    suffix = '' if normalized else '_raw'
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    psi_over_pi = cfg[select_psi + f'psi{suffix}_grid'] / np.pi

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    fig, ax = plt.subplots(1, 3, figsize=(21, 6))

    if normalized:
        x, y, z = 'dmxn_dx', 'dmyn_dx', 'dmzn_dx'
    else:
        x, y, z = 'dmx_dx', 'dmy_dx', 'dmz_dx'

    for j, eta_v in enumerate(cfg['eta_targets']):
        psi_slice = psi_over_pi[j, :]

        ax[0].plot(x_data, cfg[z][j, :], c=colors[j], linewidth=1, label=fr'{eta_v*100:.0f}%')
        ax[1].plot(x_data, cfg[y][j, :], c=colors[j], linewidth=1, label=fr'{eta_v*100:.0f}%')
        ax[2].plot(x_data, cfg[x][j, :], c=colors[j], linewidth=1, label=fr'{eta_v*100:.0f}%')

        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx]

        ax[0].scatter(x_data, cfg[z][j, :], c=point_colors)
        ax[1].scatter(x_data, cfg[y][j, :], c=point_colors)
        ax[2].scatter(x_data, cfg[x][j, :], c=point_colors)

    ax[0].set_ylabel(r'd$M_{z}$/dx')
    ax[1].set_ylabel(r'd$M_{y}$/dx')
    ax[2].set_ylabel(r'd$M_{x}$/dx')
    for a in ax:
        a.set_xlabel('x (pix)')

    tag = 'normalized' if normalized else 'un-normalized'
    fig.suptitle(f"Gradient of {cfg['sample']} {cfg['coord']}-slice — colored by {select_psi} ψ ({tag})")

    ax[0].legend(fontsize=6, ncol=2, loc='lower left', title='Fraction of local thickness')
    ax[1].legend(fontsize=6, ncol=2, loc='lower left', title='Fraction of local thickness')
    ax[2].legend(fontsize=6, ncol=2, loc='lower left', title='Fraction of local thickness')

    plt.tight_layout()
    plt.show()


for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        for normalized in [True, False]:
            plot_gradient_by_psi_bin(crop_info['dw'], select_psi='my_', normalized=normalized)
            plot_gradient_by_psi_bin(crop_info['dw'], select_psi='mx_', normalized=normalized)
            plot_gradient_by_psi_bin(crop_info['dw'], select_psi='', normalized=normalized)


In [ ]:
def plot_by_psi_bin_offset(cfg, select_psi=None, normalized=True, n_bins=11, offset=1.5):
    suffix = '' if normalized else '_raw'
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    psi_over_pi = cfg[select_psi + f'psi{suffix}_grid'] / np.pi

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    fig, ax = plt.subplots(1, 3, figsize=(21, 8))

    if normalized:
        x, y, z = 'mxn_grid', 'myn_grid', 'mzn_grid'
    else:
        x, y, z = 'mx_grid', 'my_grid', 'mz_grid'

    for j, eta_v in enumerate(cfg['eta_targets']):
        psi_slice = psi_over_pi[j, :]
        shift = j * offset

        mzn = cfg[z][j, :] + shift
        myn = cfg[y][j, :] + shift
        mxn = cfg[x][j, :] + shift

        ax[0].plot(x_data, mzn, c=colors[j], linewidth=1)
        ax[1].plot(x_data, myn, c=colors[j], linewidth=1)
        ax[2].plot(x_data, mxn, c=colors[j], linewidth=1)

        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx]

        ax[0].scatter(x_data, mzn, c=point_colors, s=20, zorder=3)
        ax[1].scatter(x_data, myn, c=point_colors, s=20, zorder=3)
        ax[2].scatter(x_data, mxn, c=point_colors, s=20, zorder=3)

        for a in ax:
            a.axhline(shift, color=colors[j], lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'{eta_v*100:.0f}%', xy=(0.15, shift), xycoords=('axes fraction', 'data'),
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=15, color=colors[j])

    ax[0].set_ylabel(r'$M_{z}$ (offset per slice)')
    ax[1].set_ylabel(r'$M_{y}$ (offset per slice)')
    ax[2].set_ylabel(r'$M_{x}$ (offset per slice)')
    for a in ax:
        a.set_xlabel('x (pix)')
        a.set_yticks([])

    tag = 'normalized' if normalized else 'un-normalized'
    fig.suptitle(f"{cfg['sample']} {cfg['coord']}-slice, colored by {select_psi} ψ ({tag})")

    plt.tight_layout()
    plt.show()


for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        #plot_by_psi_bin_offset(crop_info['dw'], select_psi='', normalized=False, offset=0.05)
        plot_by_psi_bin_offset(crop_info['dw'], select_psi='', normalized=True, offset=2)


In [ ]:
def plot_gradient_by_psi_bin_offset(cfg, select_psi='', normalized=True, n_bins=11, offset=0.25):
    suffix = '' if normalized else '_raw'
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    psi_over_pi = cfg[select_psi + f'psi{suffix}_grid'] / np.pi

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    fig, ax = plt.subplots(1, 3, figsize=(21, 8))

    if normalized:
        x, y, z = 'dmxn_dx', 'dmyn_dx', 'dmzn_dx'
    else:
        x, y, z = 'dmx_dx', 'dmy_dx', 'dmz_dx'

    for j, eta_v in enumerate(cfg['eta_targets']):
        psi_slice = psi_over_pi[j, :]
        shift = j * offset

        mzn = cfg[z][j, :] + shift
        myn = cfg[y][j, :] + shift
        mxn = cfg[x][j, :] + shift

        ax[0].plot(x_data, mzn, c=colors[j], linewidth=1)
        ax[1].plot(x_data, myn, c=colors[j], linewidth=1)
        ax[2].plot(x_data, mxn, c=colors[j], linewidth=1)

        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx]

        ax[0].scatter(x_data, mzn, c=point_colors, s=20, zorder=3)
        ax[1].scatter(x_data, myn, c=point_colors, s=20, zorder=3)
        ax[2].scatter(x_data, mxn, c=point_colors, s=20, zorder=3)

        for a in ax:
            a.axhline(shift, color='gray', lw=0.5, alpha=0.3, zorder=0)
            a.annotate(fr'{eta_v*100:.0f}%', xy=(0.15, shift), xycoords=('axes fraction', 'data'),
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=15, color=colors[j])

    ax[0].set_ylabel(r'd$M_{z}$/dx (offset per slice)')
    ax[1].set_ylabel(r'd$M_{y}$/dx (offset per slice)')
    ax[2].set_ylabel(r'd$M_{x}$/dx (offset per slice)')
    for a in ax:
        a.set_xlabel('x (pix)')
        a.set_yticks([])

    tag = 'normalized' if normalized else 'un-normalized'
    fig.suptitle(f"{cfg['sample']} {cfg['coord']}-slice, colored by {select_psi} ψ ({tag})")

    plt.tight_layout()
    plt.show()

for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        plot_gradient_by_psi_bin_offset(crop_info['dw'], select_psi='', normalized=True, offset=0.005)
        plot_gradient_by_psi_bin_offset(crop_info['dw'], select_psi='', normalized=False, offset=0.005)


In [ ]:
def plot_by_psi_sample_comparison(crop, coord, sample_ids, normalized=True, n_bins=11, offset=0.5, linestyles=None):
    suffix = '' if normalized else '_raw'
    if linestyles is None:
        linestyles = ['-', '--', ':', '-.']

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    fig, ax = plt.subplots(2, 3, figsize=(21, 16))

    if normalized:
        x, y, z = 'mxn_grid', 'myn_grid', 'mzn_grid'
        dx, dy, dz = 'dmxn_dx', 'dmyn_dx', 'dmzn_dx'
    else:
        x, y, z = 'mx_grid', 'my_grid', 'mz_grid'
        dx, dy, dz = 'dmx_dx', 'dmy_dx', 'dmz_dx'

    for s_idx, sample in enumerate(sample_ids):
        cfg = crop[sample][coord]['dw']
        x_data = cfg['unique_x'] - cfg['unique_x'].mean()
        psi_over_pi = cfg[f'psi{suffix}_grid'] / np.pi
        linestyle = linestyles[s_idx % len(linestyles)]

        for j, eta_v in enumerate(cfg['eta_targets']):
            psi_slice = psi_over_pi[j, :]
            shift = j * offset

            mzn = cfg[z][j, :] + shift
            myn = cfg[y][j, :] + shift
            mxn = cfg[x][j, :] + shift

            dmzn = cfg[dz][j, :] + shift
            dmyn = cfg[dy][j, :] + shift
            dmxn = cfg[dx][j, :] + shift

            ax[0, 0].plot(x_data, mzn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[0, 1].plot(x_data, myn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[0, 2].plot(x_data, mxn, c=colors[j], linestyle=linestyle, linewidth=1)

            ax[1, 0].plot(x_data, dmzn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[1, 1].plot(x_data, dmyn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[1, 2].plot(x_data, dmxn, c=colors[j], linestyle=linestyle, linewidth=1)

            bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
            bin_idx = np.clip(bin_idx, 0, n_bins - 1)
            point_colors = bin_colors[bin_idx]

            ax[0, 0].scatter(x_data, mzn, c=point_colors, s=20, zorder=3)
            ax[0, 1].scatter(x_data, myn, c=point_colors, s=20, zorder=3)
            ax[0, 2].scatter(x_data, mxn, c=point_colors, s=20, zorder=3)

            ax[1, 0].scatter(x_data, dmzn, c=point_colors, s=20, zorder=3)
            ax[1, 1].scatter(x_data, dmyn, c=point_colors, s=20, zorder=3)
            ax[1, 2].scatter(x_data, dmxn, c=point_colors, s=20, zorder=3)

            if s_idx == 0:
                for a in ax.flat:
                    a.axhline(shift, color='gray', lw=0.5, alpha=0.3, zorder=0)
                    a.annotate(fr'{eta_v*100:.0f}%', xy=(0.15, shift), xycoords=('axes fraction', 'data'),
                               xytext=(-8, 0), textcoords='offset points',
                               ha='right', va='center', fontsize=15, color=colors[j])

    ax[0, 0].set_ylabel(r'$M_{z}$')
    ax[0, 1].set_ylabel(r'$M_{y}$')
    ax[0, 2].set_ylabel(r'$M_{x}$')
    ax[1, 0].set_ylabel(r'd$M_{z}$/dx')
    ax[1, 1].set_ylabel(r'd$M_{y}$/dx')
    ax[1, 2].set_ylabel(r'd$M_{x}$/dx')

    sample_handles = [
        plt.Line2D([0], [0], color='black', linestyle=linestyles[i % len(linestyles)], label=sample)
        for i, sample in enumerate(sample_ids)
    ]

    for a in ax.flat:
        a.set_xlabel('x (pix)')
        a.set_yticks([])
        a.legend(handles=sample_handles, fontsize=15, loc='upper right', title='sample')

    tag = 'normalized' if normalized else 'un-normalized'
    fig.suptitle(f'Sample comparison {coord}-slice ({tag})')
    plt.tight_layout()
    plt.show()


for coord in ['x', 'y']:
    for normalized in [True, False]:
        plot_by_psi_sample_comparison(crop1, coord, sample_ids, normalized=normalized)


In [ ]:
def plot_by_psi_sample_comparison_whighlight(crop, coord, sample_ids, normalized=True, n_bins=11, offset=0.05, psi_tol=0.1, linestyles=None): #psi_tol=0.05
    suffix = '' if normalized else '_raw'
    if linestyles is None:
        linestyles = ['-', '--', ':', '-.']

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    fig, ax = plt.subplots(2, 3, figsize=(21, 16))

    if normalized:
        x, y, z = 'mxn_grid', 'myn_grid', 'mzn_grid'
        dx, dy, dz = 'dmxn_dx', 'dmyn_dx', 'dmzn_dx'
    else:
        x, y, z = 'mx_grid', 'my_grid', 'mz_grid'
        dx, dy, dz = 'dmx_dx', 'dmy_dx', 'dmz_dx'

    for s_idx, sample in enumerate(sample_ids):
        cfg = crop[sample][coord]['dw']
        x_data = cfg['unique_x'] - cfg['unique_x'].mean()
        psi_over_pi = cfg[f'psi{suffix}_grid'] / np.pi
        linestyle = linestyles[s_idx % len(linestyles)]

        for j, eta_v in enumerate(cfg['eta_targets']):
            psi_slice = psi_over_pi[j, :]
            shift = j * offset

            mzn = cfg[z][j, :] + shift
            myn = cfg[y][j, :] + shift
            mxn = cfg[x][j, :] + shift

            dmzn = cfg[dz][j, :] + shift
            dmyn = cfg[dy][j, :] + shift
            dmxn = cfg[dx][j, :] + shift

            ax[0, 0].plot(x_data, mzn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[0, 1].plot(x_data, myn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[0, 2].plot(x_data, mxn, c=colors[j], linestyle=linestyle, linewidth=1)

            ax[1, 0].plot(x_data, dmzn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[1, 1].plot(x_data, dmyn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[1, 2].plot(x_data, dmxn, c=colors[j], linestyle=linestyle, linewidth=1)

            bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
            bin_idx = np.clip(bin_idx, 0, n_bins - 1)
            point_colors = bin_colors[bin_idx]
            
            bloch_mask = np.isclose(np.abs(psi_slice), 0.5, atol=psi_tol)
            neel_mask = (np.abs(psi_slice) <= psi_tol) | (np.isclose(np.abs(psi_slice), 1, atol=psi_tol))
            
            point_colors[neel_mask] = mcolors.to_rgba('cyan')
            point_colors[bloch_mask] = mcolors.to_rgba('magenta')

            ax[0, 0].scatter(x_data, mzn, c=point_colors, s=20, zorder=3)
            ax[0, 1].scatter(x_data, myn, c=point_colors, s=20, zorder=3)
            ax[0, 2].scatter(x_data, mxn, c=point_colors, s=20, zorder=3)

            ax[1, 0].scatter(x_data, dmzn, c=point_colors, s=20, zorder=3)
            ax[1, 1].scatter(x_data, dmyn, c=point_colors, s=20, zorder=3)
            ax[1, 2].scatter(x_data, dmxn, c=point_colors, s=20, zorder=3)

            if s_idx == 0:
                for a in ax.flat:
                    a.axhline(shift, color=colors[j], lw=1, alpha=0.3, zorder=0)
                    a.annotate(fr'{eta_v*100:.0f}%', xy=(0.15, shift), xycoords=('axes fraction', 'data'),
                               xytext=(-8, 0), textcoords='offset points',
                               ha='right', va='center', fontsize=15, color=colors[j])

    ax[0, 0].set_ylabel(r'$M_{z}$')
    ax[0, 1].set_ylabel(r'$M_{y}$')
    ax[0, 2].set_ylabel(r'$M_{x}$')
    ax[1, 0].set_ylabel(r'd$M_{z}$/dx')
    ax[1, 1].set_ylabel(r'd$M_{y}$/dx')
    ax[1, 2].set_ylabel(r'd$M_{x}$/dx')

    sample_handles = [
        plt.Line2D([0], [0], color='black', linestyle=linestyles[i % len(linestyles)], label=sample)
        for i, sample in enumerate(sample_ids)
    ]

    for a in ax.flat:
        a.set_xlabel('x (pix)')
        a.set_yticks([])
        a.legend(handles=sample_handles, fontsize=15, loc='upper right', title='sample')

    tag = 'normalized' if normalized else 'un-normalized'
    fig.suptitle(f'Sample comparison {coord}-slice ({tag})')
    plt.tight_layout()
    plt.show()



for coord in ['x', 'y']:
    for normalized in [True, False]:
        plot_by_psi_sample_comparison_whighlight(crop1, coord, sample_ids, normalized=normalized, offset=2, psi_tol = 0.01)


In [ ]:
crop1.keys()


In [ ]:

for sample, coords in crop1.items():
    print (sample)
    # print(coords)
    if sample == '665':
        for coord, data_dict in coords.items():
            vis.calc_neel_bloch_stats(crop1[sample][coord]['dw'], normalized=True)
            vis.calc_neel_bloch_stats(crop1[sample][coord]['dw'], normalized=False)

            vis.plot_psi_bin_offset_w_highlight(crop1[sample][coord]['dw'], select_psi='', normalized=True, n_bins=11, offset=2.5, colors=colors)
            vis.plot_psi_bin_offset_w_highlight(crop1[sample][coord]['dw'], select_psi='', normalized=False, n_bins=11, offset=0.03, colors=colors)

    else: 
        continue

In [ ]:
# calc_neel_bloch_stats stores neel/bloch percentages on `cfg` -- once for
# the normalized branch (unsuffixed keys) and once for the un-normalized
# branch ('_raw'-suffixed keys). This has to run before
# vis.plot_psi_bin_offset_w_highlight, which only *reads* those stats.
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        vis.calc_neel_bloch_stats(samples[sample][coord], normalized=True)
        vis.calc_neel_bloch_stats(samples[sample][coord], normalized=False)

        vis.plot_psi_bin_offset_w_highlight(samples[sample][coord], select_psi='', normalized=True, n_bins=11, offset=2.5, colors=colors)
        vis.plot_psi_bin_offset_w_highlight(samples[sample][coord], select_psi='', normalized=False, n_bins=11, offset=2.5, colors=colors)


In [ ]:
# Final normalized / un-normalized Neel vs Bloch summary (4-panel: y per-eta,
# y overall bar, x per-eta, x overall bar), shown once per branch.
vis.plot_neel_bloch_comparison(samples, sample_ids, normalized=True)
vis.plot_neel_bloch_comparison(samples, sample_ids, normalized=False)


# Compare normalized vs. un-normalized results: Bloch vs. Néel

In [ ]:
# Side-by-side bar comparison of the overall % Neel / % Bloch computed from
# the NORMALIZED vector components vs the UN-NORMALIZED components.
vis.plot_neel_bloch_norm_vs_raw(samples, sample_ids)


In [ ]:
# Numeric summary: for each sample & slice direction, how much does
# normalizing the vector components change the Neel/Bloch classification?
print(f"{'sample':>6} {'coord':>5} {'%Neel (norm)':>13} {'%Neel (raw)':>12} {'diff':>7}   "
      f"{'%Bloch (norm)':>14} {'%Bloch (raw)':>13} {'diff':>7}")
for sample in sample_ids:
    for coord in ['y', 'x']:
        cfg = samples[sample][coord]
        neel_n, neel_r = cfg['neel_pct_overall'], cfg['neel_pct_overall_raw']
        bloch_n, bloch_r = cfg['bloch_pct_overall'], cfg['bloch_pct_overall_raw']
        print(f"{sample:>6} {coord:>5} {neel_n:13.2f} {neel_r:12.2f} {neel_n-neel_r:7.2f}   "
              f"{bloch_n:14.2f} {bloch_r:13.2f} {bloch_n-bloch_r:7.2f}")


In [ ]:
# Direct scatter comparison of psi computed from normalized vs un-normalized
# components -- if normalization has little effect, points should sit close
# to the y=x line.
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for coord, a in zip(['y', 'x'], ax):
    for sample, color in zip(samples, sample_colors):
        cfg = samples[sample][coord]
        psi_norm = (cfg['psi_grid'] / np.pi).ravel()
        psi_raw = (cfg['psi_raw_grid'] / np.pi).ravel()
        a.scatter(psi_norm, psi_raw, s=6, alpha=0.4, color=color, label=sample)

    a.plot([-1, 1], [-1, 1], color='k', lw=1, ls='--', label='y = x')
    a.set_xlabel(r'$\psi$ / $\pi$ (from normalized components)')
    a.set_ylabel(r'$\psi$ / $\pi$ (from un-normalized components)')
    a.set_title(f'{coord}-slice: ψ agreement')
    a.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()


# Calculate spatial energy variation like in Legrand

In [ ]:
samples['665']['y'].keys()


In [ ]:
# vis.calc_energies now always computes BOTH the normalized branch
# (exchange_grid, dmi_grid, pma_grid -- unchanged names) and the
# un-normalized branch (exchange_raw_grid, dmi_raw_grid, pma_raw_grid)
# in a single call.
for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        crop1[sample][coord]['dw'] = vis.calc_energies(crop_info['dw'], coord, A=1.0, D=1.0, K=1.0)


In [ ]:
for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        vis.plot_energies(crop_info['dw'], sample, coord=coord, offset=1, normalized=True, colors=colors)
        vis.plot_energies(crop_info['dw'], sample, coord=coord, offset=1, normalized=False, colors=colors)


In [ ]:
for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        print(sample, coord, type(crop_info), list(crop_info.keys()) if isinstance(crop_info, dict) else crop_info)


# Compare to basic fits

In [ ]:
def plot_profile_with_model(data_dict, wall_types, x0, delta_demo, field_pairs):
    field_to_comp = {'mz': 'Mz', 'my': 'My', 'mx': 'Mx'}

    x_data = data_dict['unique_x']
    x_data = x_data - x_data.mean()

    for field, field_n in field_pairs:
        comp = field_to_comp[field]

        fig, ax1 = plt.subplots(1, 2, figsize=(15, 5))

        # --- experimental data (normalized branch, which is the physically
        # bounded [-1, 1] quantity that's comparable to the M/Ms model) ---
        for j, eta_v in enumerate(data_dict['eta_targets']):
            ax1[0].plot(x_data, data_dict[field_n + '_grid'][j, :],
                        color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')
            ax1[1].plot(x_data, data_dict['d' + field_n + '_dx'][j, :],
                        color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')

        # --- overlay theoretical wall model(s), computed fresh for this x0 ---
        for wall, wcfg in wall_types.items():
            psi = wcfg['psi']
            if np.isscalar(psi):
                Mx, My, Mz = vis.magnetization_components(x0, delta_demo, psi)
                values = {'Mx': Mx, 'My': My, 'Mz': Mz}[comp]
                dvalues = np.gradient(values, x0[1] - x0[0])

                ax1[0].plot(x0, values, color=wcfg['color'], lw=2,
                            label=f'{wall} model, ψ={psi:.2f} & Δ = {delta_demo*8:.2f} nm')
                ax1[1].plot(x0, dvalues, color=wcfg['color'], lw=2,
                            label=f'{wall} model, ψ={psi:.2f} & Δ = {delta_demo*8:.2f} nm')
            else:
                for psi_val, color in zip(psi, wcfg['colors']):
                    Mx, My, Mz = vis.magnetization_components(x0, delta_demo, psi_val)
                    values = {'Mx': -Mx, 'My': My, 'Mz': Mz}[comp]  # keep your -Mx flip for hybrid
                    dvalues = np.gradient(values, x0[1] - x0[0])
                    label = rf'{wall} $\psi={np.degrees(psi_val):.0f}°$'
                    ax1[0].plot(x0, values, color=color, ls='--', lw=1, label=label)
                    ax1[1].plot(x0, dvalues, color=color, ls='--', lw=1, label=label)

        ax1[0].set_ylabel(r'$M$')
        ax1[0].set_xlabel('x (pix), centered')
        ax1[0].legend(fontsize=7, ncol=2, title='data / model', loc='upper right')
        ax1[0].set_title(f"{field} \n full {data_dict['coord']}-DW profile + model")

        ax1[1].set_xlabel('x (pix), centered')
        ax1[1].set_ylabel(r'$dM(x)/dx$')
        ax1[1].legend(fontsize=7, ncol=2, title='data / model', loc='upper right')
        ax1[1].set_title(f"{field} \n full {data_dict['coord']}-DW Gradient + model")

        plt.tight_layout()
        plt.show()

test = ['665', '270', '452']

for sample in test:
    xmax, xmin = crop1[sample]['x']['xmax'], crop1[sample]['x']['xmin']
    x0 = np.linspace(0, (xmax - xmin), (xmax - xmin) + 1)
    x0 = x0 - x0.mean()  # recenter around 0
    delta = np.mean(dw_fits[sample]['x']['normalized']['dw_width']) / 8  # pixel size

    plot_profile_with_model(crop1[sample]['x']['dw'], wall_types, x0, delta, field_pairs)


# Update wall profile fit

In [ ]:
## Update hybrid fit, using an explicit sample/coord so this cell doesn't
## depend on whatever loop iteration happened to run last (that's what the
## original notebook was doing here, relying on bare globals like
## `mzn_grid`/`dx0`/`set_sigma` left over from earlier cells -- fragile).
## Data referenced here already reflects SMOOTH_DATA/SMOOTH_SIGMA from the
## User setup cell, so there's no need to smooth again here.

fit_sample, fit_coord = '452', 'x'

wall = crop1[fit_sample][fit_coord]['dw']
fit = dw_fits[fit_sample][fit_coord]

pixel_size_nm = 8
delta = np.array(fit['normalized']['dw_width']) / pixel_size_nm
center = np.array(fit['normalized']['dw_center']) / pixel_size_nm

x0 = wall['unique_x'] - wall['unique_x'].mean()
dx0 = x0[1] - x0[0]
eta_targets = wall['eta_targets']

n_slices = len(eta_targets)
psi_hybrid_list = np.linspace(-np.pi / 9, np.pi / 4, n_slices)

colors2 = plt.cm.viridis(np.linspace(0, 1, len(center)))


def magnetization_components_w_center(x, delta, psi, center):
    """
    Compute Mx, My, Mz for a domain wall with:
      - wall normal along x
      - easy axis along z
      - azimuthal angle psi (psi=0: Neel, psi=pi/2: Bloch)
    Returns normalized components (M/Ms).
    """
    theta = vis.theta_profile(x, delta)
    Mx = np.sin(theta) * np.cos(psi)
    My = np.sin(theta) * np.sin(psi)
    Mz = np.cos(theta)
    return Mx, My, Mz


for label, field_key, ylabel in [
    ('Mz', 'mzn_grid', r'$M_z / M_{norm}$'),
    ('My', 'myn_grid', r'$M_y / M_{norm}$'),
    ('Mx', 'mxn_grid', r'$M_x / M_{norm}$'),
]:
    fig, ax1 = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'Updated Hybrid Wall fit from Hyperbolic Tan Fit Parameters ({fit_sample}, {fit_coord}-slice)', fontsize=18, y=1.0)

    for j, eta_v in enumerate(eta_targets):
        m_component = wall[field_key][j, :]
        dM_dx = np.gradient(m_component, dx0)

        Mx_h, My_h, Mz_h = magnetization_components_w_center(x0, delta[j], psi_hybrid_list[j], center[j])
        model_val = {'Mx': Mx_h, 'My': My_h, 'Mz': Mz_h}[label]
        if label == 'Mx' and j == 0:
            model_val = -model_val
        dModel_dx = np.gradient(model_val, dx0)

        ax1[0].plot(x0, m_component, color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')
        ax1[1].plot(x0, dM_dx, color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')

        ax1[0].plot(x0 + center[j], model_val, color=colors2[j], linestyle=':')
        ax1[1].plot(x0 + center[j], dModel_dx, color=colors2[j], linestyle=':')

    ax1[0].set_ylabel(ylabel)
    ax1[0].set_xlabel('x (pix)')
    ax1[0].legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc='upper right')
    ax1[0].set_title('Domain Wall Profile' + (' (smoothed)' if SMOOTH_DATA else ''))

    ax1[1].set_xlabel('x (pix)')
    ax1[1].set_ylabel(fr'$d{label}(x)$/dx')
    ax1[1].legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc='upper right')
    ax1[1].set_title('Gradient' + (' (smoothed)' if SMOOTH_DATA else ''))

    plt.tight_layout()
    plt.show()
